# Section 2 — Concept attention (remote on NDIF)

The model in this section is **FLUX.2-klein-4B** — too big to fit in a
Colab T4's 16 GB. Instead of loading weights locally, we instantiate
it with `dispatch=False`, which gives us only the meta-tensor skeleton
(the module graph + names) needed to write `nnsight` interventions.
The actual forward pass runs on **NDIF**, the workshop-hosted
inference cluster, by passing `remote=True` to each `.trace()` /
`.generate()` call.

**The technique** ([ConceptAttention, Helbling et al. CVPR 2025](https://arxiv.org/abs/2502.04320)):
augment the joint text+image attention with a third "concept" stream.
Encode a list of concept words (e.g. `["cat", "grass", "sky", "tree"]`)
through the same text encoder as the prompt, splice their per-token
embeddings into `encoder_hidden_states`, and install an attention mask
that two-way-isolates them:

  * non-concept queries cannot attend to concept keys  → the image
    stream stays bit-identical to a vanilla forward (the concepts are
    invisible to it).
  * concept queries cannot attend to prompt  keys      → the concept
    stream attends only to {concept, image}, matching the paper's
    separate-attention setup.

For each double-stream block, take the inner product
`einsum(image_post_attn, concept_post_attn)`, softmax across concepts,
and average over a few selected layers + timesteps. The result is a
per-concept spatial heatmap that tells you which patches "belong to"
each concept.

All of the steps below run remotely — the Colab kernel only does
tokenisation, a handful of tensor concats, and the final colorisation.

In [ ]:
import torch
from nnsight import DiffusionModel

from pathlib import Path

# No CONFIG.API.HOST, no NDIF, no network calls at all.
# dispatch=True loads the real weights locally right now (instead of
# staying on meta and running remotely), so make sure download_flux.py
# has already been run.

DEVICE = "cuda"           # or "cuda", "cpu", etc.
LOCAL_DIR = Path("models/flux2-klein-4b")

flux = DiffusionModel(
    LOCAL_DIR,
    dispatch=True,
    torch_dtype=torch.bfloat16,  # drop/adjust if VRAM constrained
    device_map=DEVICE,           # or "cuda", "cpu", etc.
)

PROMPT_2 = "A cat in a park on the grass by a tree"
CONCEPTS = ["cat", "grass", "sky", "tree"]
NUM_INFERENCE_STEPS_2 = 4
SEED_2 = 0

## Encode prompt + concepts on NDIF

We need the joint `[prompt | concepts]` sequence embedding the
transformer will receive. A `model.session(remote=True)` ships an
arbitrary block of Python to NDIF as a single payload — inside it,
we just call `flux.pipeline.encode_prompt(...)` directly, the same
way we would locally, but the underlying text-encoder forward runs
remotely. No diffusion forward, no inner `.trace(...)` per text —
just five calls to the pipeline's own encoder helper.

For Qwen3 chat-templated inputs the actual concept word lives at
position 3 (positions 0..2 are `<|im_start|>user\n`, 4..12 are the
chat-suffix, 13..511 are padding), so we slice that one position out
of each concept's encoding.

In [ ]:
import nnsight  
with flux.session():
    # Encode the prompt — return value is (prompt_embeds, text_ids).
    prompt_embeds = nnsight.save(flux.pipeline.encode_prompt(
        prompt=PROMPT_2,
        device="cuda",
        num_images_per_prompt=1,
    )[0])

    # Encode each concept and keep just the position-3 (concept word) row.
    concept_rows = nnsight.save(list())
    for c in CONCEPTS:
        emb_ = flux.pipeline.encode_prompt(
            prompt=c,
            device="cuda",
            num_images_per_prompt=1,
        )[0]
        emb = nnsight.save(emb_)
        concept_rows.append(emb[:, 3:4, :])

print("prompt_embeds shape: ", tuple(prompt_embeds.shape))
print(
    "concept rows:        ", len(concept_rows), "× shape", tuple(concept_rows[0].shape)
)

concept_embeds = torch.cat(concept_rows, dim=1)  # [1, L_c, D]

# Splice: prompt first, concepts at the tail of the encoder sequence.
prompt_embeds_full = torch.cat(
    [prompt_embeds, concept_embeds.to(prompt_embeds.dtype)],
    dim=1,
)
L_txt = prompt_embeds.shape[1]
L_c = concept_embeds.shape[1]
L_img = (1024 // 16) ** 2  # 64 × 64 = 4096

## Build attention mask + zero-position txt_ids

The two off-diagonal blocks of the mask are what makes this work — see
the markdown above for what each one buys us.

FLUX.2's `_prepare_text_ids` would otherwise number concept positions
1..L_c on the RoPE L axis. The paper specifies `concept_pe = 0`, so
we pre-build the txt_ids with concept rows zeroed and override the
pipeline-generated version per denoising step.

In [ ]:
allow = torch.ones(L_txt + L_c + L_img, L_txt + L_c + L_img, dtype=torch.bool)
c_start, c_end = L_txt, L_txt + L_c
allow[:c_start, c_start:c_end] = False  # prompt  → concept blocked
allow[c_end:, c_start:c_end] = False  # image   → concept blocked
allow[c_start:c_end, :c_start] = False  # concept → prompt  blocked

text_ids = torch.zeros(1, L_txt + L_c, 4, dtype=torch.long)
text_ids[:, :L_txt, 3] = torch.arange(L_txt)

## Trace the generation, capture per-block scores

Inside the trace we accumulate the per-(step, layer) concept-image
softmax score in-place — `score_acc` is a saved tensor we add to at
every double block at every selected denoising step.

The accumulator never materialises the full
`[T, L, B, num_patches, D]` activation tensor; it stays at
`[1, num_concepts, num_image_patches]` regardless of how many steps or
layers we average over.

In [ ]:
with flux.generate(
    prompt_embeds=prompt_embeds_full,
    attention_kwargs={"attention_mask": allow},
    width=1024,
    height=1024,
    num_inference_steps=NUM_INFERENCE_STEPS_2,
    seed=SEED_2,
    remote=False,
) as tracer:
    # CPU accumulator — works whether captured tensors come back from
    # NDIF on cuda:0 (remote GPU) or end up local; we just `.cpu()` each
    # score before adding. Pay one device-transfer per (step × layer),
    # not per residual.
    score_acc = torch.zeros(1, L_c, L_img, dtype=torch.float32).save()

    for _step in tracer.iter[:]:
        # Override the pipeline-derived txt_ids with our zero-position version.
        new_kwargs = dict(flux.transformer.inputs[1])
        new_kwargs["txt_ids"] = text_ids
        flux.transformer.inputs = (flux.transformer.inputs[0], new_kwargs)

        # Capture per-block PRE-projection attention outputs in forward order:
        # `to_add_out` (encoder) is called BEFORE `to_out[0]` (image) in
        # Flux2AttnProcessor — access in that order for nnsight's one-shot hooks.
        for blk in flux.transformer.transformer_blocks:
            enc_attn_pre = blk.attn.to_add_out.inputs[0][0]  # [1, L_txt+L_c, D]
            img_attn_pre = blk.attn.to_out[0].inputs[0][0]  # [1, L_img,     D]
            concept_pre = enc_attn_pre[:, L_txt:]  # [1, L_c, D]
            scores = torch.einsum(
                "bpd,bcd->bcp",
                img_attn_pre.float(),
                concept_pre.float(),
            ).softmax(
                dim=-2
            )  # [1, L_c, L_img]
            score_acc.add_(scores.cpu())

    result = tracer.result.save()

## Display the image and per-concept heatmaps

In [ ]:
n_blocks = len(flux.transformer.transformer_blocks)
n_accumulated = NUM_INFERENCE_STEPS_2 * n_blocks

util.show_concept_heatmaps(
    image=result.images[0],
    score_acc=score_acc,
    n_accumulated=n_accumulated,
    concepts=CONCEPTS,
)